# 02 - Data Cleaning: Online Retail

This notebook continues from notebook 01. Here I clean the raw data based on the
issues I found earlier, so I end up with a table of real sales that I can use for RFM.

## 1. Load the data

In [1]:
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

DATA_PATH = "../data/raw/Online Retail.xlsx"
retail_data = pd.read_excel(DATA_PATH)

retail_data.shape

(541909, 8)

The raw data has 541,909 rows and 8 columns, the same as in notebook 01.

## 2. Basic prep

A few columns need a quick fix before I start cleaning. `InvoiceNo` mixes numbers and
letters (`C` for cancellations, `A` for bad debt), so it should be a string. `StockCode`
can have lowercase letters or extra spaces, which would split one product into two codes.
`Description` can also have stray spaces, and `CustomerID` is loaded as a float only
because it has missing values.

I do all of this on a copy called `clean`, so `retail_data` stays untouched and I can
compare the two later.

In [2]:
clean = retail_data.copy()
clean["InvoiceNo"] = clean["InvoiceNo"].astype("string")
clean["CustomerID"] = clean["CustomerID"].astype("Int64")
clean["StockCode"] = clean["StockCode"].astype("string").str.strip().str.upper()
clean["Description"] = clean["Description"].astype("string").str.strip()

In [3]:
changes_codes = (retail_data["StockCode"].astype("string") != clean["StockCode"]).sum()
print(f"Changed {changes_codes} StockCode values to uppercase and stripped whitespace.")

Changed 1977 StockCode values to uppercase and stripped whitespace.


1,977 `StockCode` values changed after the cleanup. They only differed from the original
by letter case or a stray space, so without this fix the same product would be counted
under two different codes.

In [4]:
clean.shape

(541909, 8)

In [5]:
clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  string        
 1   StockCode    541909 non-null  string        
 2   Description  540455 non-null  string        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  Int64         
 7   Country      541909 non-null  str           
dtypes: Int64(1), datetime64[us](1), float64(1), int64(1), str(1), string(3)
memory usage: 33.6 MB


`InvoiceNo`, `StockCode` and `Description` are now strings and `CustomerID` is `Int64`,
so the missing values stay empty instead of forcing the column to float. The shape is
still (541909, 8), so no rows were lost. The non-null counts also match notebook 01:
`Description` has 540,455 values (1,454 missing) and `CustomerID` has 406,829
(135,080 missing). One `Description` value (`20713`, on a stock adjustment row) is stored as a number in the Excel file. I convert the column to string before stripping, because otherwise `.str.strip()` would turn that value into a missing one.

## 3. Cleaning plan

These are the issues I found in notebook 01 and what I plan to do with each one. The plan can still change if I find something new while cleaning.

| # | Issue | Decision | Reason |
|---|---|---|---|
| 1 | Missing `CustomerID` | Keep | These are still valid sales. I only filter them out later for RFM, because RFM needs a customer. |
| 2 | Missing `Description` | Drop | Most of these rows already fall under other rules (zero price, stock adjustments), so I'll check what is left after those steps. |
| 3 | Exact duplicate rows | Drop | Identical lines on the same invoice and minute look like double entries. Their share of revenue should be tiny, and Frequency is not affected because it counts unique invoices. |
| 4 | Cancellations (`C` invoices) | Separate | They are real transactions that belong to a customer and reduce what the customer really spent, so I keep them in their own table. |
| 5 | Stock adjustments (negative `Quantity`, not `C`, price 0) | Drop | They look like internal warehouse notes (damaged, check, thrown away) with no customer and no price, not purchases. |
| 6 | `A` invoices ("Adjust bad debt") | Drop | These are accounting entries, not product sales, and two of them have a negative price of about -11,062. |
| 7 | Non-product `StockCode` (`POST`, `DOT`, `M`, ...) | Drop | Postage, fees and manual entries are not products, and they would mess up any product analysis. I keep them in an excluded table for reference. |
| 8 | Zero price with positive `Quantity` | Drop | Free items or samples. They count as a transaction but bring in no money, which would distort Frequency and price statistics. |
| 9 | Extreme order-and-cancel pairs and the 12,540-unit invoice | Drop | Each pair cancels itself out, but the positive side would inflate revenue a lot (80,995 units at 2.08 is about 168k). The 12,540-unit invoice is already covered by the zero-price rule. |
| 10 | Several `Description` values per `StockCode` | Fix | I use the most common description for each code, since `StockCode` is the real key for a product. |
| 11 | December 2011 is incomplete | Keep | The rows are valid, so this is not a cleaning problem. I handle it later when choosing the reference date for Recency and when reading monthly trends. |
| 12 | `CustomerID` stored as float | Fix | Already fixed in section 2 by converting it to `Int64`. |

Some rows match more than one rule (a cancelled postage fee is both a cancellation and a non-product code, for example). I apply the rules in this order: bad debt, extreme pairs, cancellations, stock adjustments, non-product codes, zero price. That way each row ends up in only one group.

## 4. Apply the cleaning rules

I apply the rules from the plan one at a time, in the same order as in the plan. I keep a table called `remaining` with the rows that have not been assigned to any group yet. Each rule moves the matching rows out of `remaining` into their own table, so a row can only end up in one place and nothing is deleted by accident. After every step I note how many rows moved and how many are left in `cleaning_log`.

In [6]:
remaining = clean.copy()
cleaning_log = []
remaining.shape

(541909, 8)

### 4a. Bad debt invoices

3 rows start with `A`, and all of them are "Adjust bad debt" lines with no customer. Two have a negative `UnitPrice` of about -11,062 and one is positive, so they roughly cancel each other out. After moving them, 541,906 rows are left in `remaining`.

In [7]:
is_bad_debt = remaining["InvoiceNo"].str.startswith("A")
remaining[is_bad_debt][["InvoiceNo", "Description", "Quantity", "UnitPrice", "CustomerID"]]

,InvoiceNo,Description,Quantity,UnitPrice,CustomerID
299982,A563185,Adjust bad debt,1,11062.06,<NA>
299983,A563186,Adjust bad debt,1,-11062.06,<NA>
299984,A563187,Adjust bad debt,1,-11062.06,<NA>


In [8]:
bad_debt = remaining[is_bad_debt].copy()
remaining = remaining[~is_bad_debt].copy()

cleaning_log.append({
    "step": "bad debt invoices removed",
    "rows_moved": len(bad_debt),
    "rows_remaining": len(remaining),
})

assert len(bad_debt) + len(remaining) == len(clean), "rows lost or counted twice"
print(len(bad_debt), len(remaining))

3 541906


3 rows start with `A`, and all of them are "Adjust bad debt" lines with no customer. Two have a negative `UnitPrice` of about -11,062 and one is positive, so they roughly cancel each other out. After moving them, 541906 rows are left in `remaining`.

### 4b. Extreme order-and-cancel pairs

Two orders have a huge quantity and were cancelled right away with exactly the same quantity: 541431 / C541433 (74,215 units, customer 12346) and 581483 / C581484 (80,995 units, customer 16446). Each pair cancels itself out, but the positive side would still inflate revenue a lot, so I move both sides of each pair out together.

In [9]:
EXTREME_PAIR_INVOICES = ["541431", "C541433", "581483", "C581484"]
is_extreme_pair = remaining["InvoiceNo"].isin(EXTREME_PAIR_INVOICES)
remaining[is_extreme_pair][["InvoiceNo", "Description", "Quantity", "UnitPrice", "CustomerID"]]

,InvoiceNo,Description,Quantity,UnitPrice,CustomerID
61619,541431,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,12346
61624,C541433,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,12346
540421,581483,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,16446
540422,C581484,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08,16446


In [10]:
rows_before = len(remaining)

extreme_pairs = remaining[is_extreme_pair].copy()
remaining = remaining[~is_extreme_pair].copy()

cleaning_log.append({
    "step": "extreme order-and-cancel pairs removed",
    "rows_moved": len(extreme_pairs),
    "rows_remaining": len(remaining),
})

assert len(extreme_pairs) + len(remaining) == rows_before, "rows lost or counted twice"
assert extreme_pairs["Quantity"].sum() == 0, "the pairs do not cancel each other out"
print(len(extreme_pairs), len(remaining))

4 541902


4 rows moved, and their `Quantity` adds up to 0, which confirms that each order was cancelled in full. 541,902 rows are left in `remaining`.

### 4c. Cancellations

Invoices starting with `C` are cancellations and have a negative `Quantity`. I don't drop them like the other groups, because they are real transactions that belong to a customer and reduce what that customer really spent. So I move them to their own table, `cancellations`, and decide later how to use them for the Monetary part of RFM.

In [11]:
is_cancellation = remaining["InvoiceNo"].str.startswith("C")
remaining[is_cancellation][["InvoiceNo", "Description", "Quantity", "UnitPrice", "CustomerID"]].head(10)

,InvoiceNo,Description,Quantity,UnitPrice,CustomerID
141,C536379,Discount,-1,27.50,14527
154,C536383,SET OF 3 COLOURED FLYING DUCKS,-1,4.65,15311
235,C536391,PLASTERS IN TIN CIRCUS PARADE,-12,1.65,17548
236,C536391,PACK OF 12 PINK PAISLEY TISSUES,-24,0.29,17548
237,C536391,PACK OF 12 BLUE PAISLEY TISSUES,-24,0.29,17548
238,C536391,PACK OF 12 RED RETROSPOT TISSUES,-24,0.29,17548
239,C536391,CHICK GREY HOT WATER BOTTLE,-12,3.45,17548
240,C536391,PLASTERS IN TIN VINTAGE PAISLEY,-12,1.65,17548
241,C536391,PLASTERS IN TIN SKULLS,-24,1.65,17548
939,C536506,JAM MAKING SET WITH JARS,-6,4.25,17897


In [12]:
rows_before = len(remaining)
cancellations = remaining[is_cancellation].copy()
remaining = remaining[~is_cancellation].copy()
cleaning_log.append({
    "step": "cancellations separated",
    "rows_moved": len(cancellations),
    "rows_remaining": len(remaining),
})
assert len(cancellations) + len(remaining) == rows_before, "rows lost or counted twice"
assert (cancellations["Quantity"] < 0).all(), "some cancellations have a non-negative Quantity"
print(len(cancellations), len(remaining))

9286 532616


9,286 cancellation rows moved to their own table, and all of them have a negative `Quantity`, so the `C` rule matches the sign. That is 2 fewer than in notebook 01 because two cancellations were already moved with the extreme pairs. 532,616 rows are left in `remaining`.

### 4d. Stock adjustments

All the `C` invoices are already out of `remaining`, so a negative `Quantity` here can no longer be a cancellation. In notebook 01 I saw that these rows look like internal warehouse notes, not purchases. I move them out, and I add checks to prove that they have no customer and no price.

In [13]:
is_stock_adjustment = remaining["Quantity"] < 0
print(is_stock_adjustment.sum(), "rows with negative Quantity (stock adjustments) found")
remaining[is_stock_adjustment][["InvoiceNo", "Description", "Quantity", "UnitPrice", "CustomerID"]].head(10)

1336 rows with negative Quantity (stock adjustments) found


,InvoiceNo,Description,Quantity,UnitPrice,CustomerID
2406,536589,<NA>,-10,0.0,<NA>
4347,536764,<NA>,-38,0.0,<NA>
7188,536996,<NA>,-20,0.0,<NA>
7189,536997,<NA>,-20,0.0,<NA>
7190,536998,<NA>,-6,0.0,<NA>
7192,537000,<NA>,-22,0.0,<NA>
7193,537001,<NA>,-6,0.0,<NA>
7195,537003,<NA>,-2,0.0,<NA>
7196,537004,<NA>,-30,0.0,<NA>
7197,537005,<NA>,-70,0.0,<NA>


In [14]:
rows_before = len(remaining)

stock_adjustments = remaining[is_stock_adjustment].copy()

assert not stock_adjustments["InvoiceNo"].str.startswith("C").any(), "some stock adjustments are cancellations"
assert (stock_adjustments["UnitPrice"] == 0).all(), "some stock adjustments have a non-zero UnitPrice"
assert stock_adjustments["CustomerID"].isnull().all(), "some stock adjustments have a CustomerID"

remaining = remaining[~is_stock_adjustment].copy()

cleaning_log.append({
    "step": "stock adjustments removed",
    "rows_moved": len(stock_adjustments),
    "rows_remaining": len(remaining),
})

assert len(stock_adjustments) + len(remaining) == rows_before, "rows lost or counted twice"
print(len(stock_adjustments), len(remaining))

1336 531280


1,336 rows moved. All of them have `UnitPrice` 0 and no `CustomerID`, and none of them is a `C` invoice, which matches what I found in notebook 01. 531,280 rows are left in `remaining`.

### 4e. Non-product stock codes

A normal product code starts with five digits (for example `85123A`). Codes like `POST`, `DOT` or `M` are postage, fees and manual entries, not products, and they would mess up any product analysis. I find them with a pattern that checks whether the code starts with five digits, and I move the ones that don't out of `remaining`.

In [15]:
is_not_product = ~remaining["StockCode"].str.match(r"^\d{5}")
print(is_not_product.sum(), "rows with non-product StockCode found")
remaining[is_not_product][["InvoiceNo", "Description", "Quantity", "UnitPrice", "CustomerID"]].head(10)

2396 rows with non-product StockCode found


,InvoiceNo,Description,Quantity,UnitPrice,CustomerID
45,536370,POSTAGE,3,18.00,12583
386,536403,POSTAGE,1,15.00,12791
1123,536527,POSTAGE,1,18.00,12662
1423,536540,CARRIAGE,1,50.00,14911
1814,536544,DOTCOM POSTAGE,1,569.77,<NA>
2239,536569,Manual,1,1.25,16274
2250,536569,Manual,1,18.95,16274
3041,536592,DOTCOM POSTAGE,1,607.49,<NA>
4406,536779,Bank Charges,1,15.00,15823
5073,536840,POSTAGE,1,18.00,12738


In [16]:
rows_before = len(remaining)
non_products = remaining[is_not_product].copy()

assert not non_products["StockCode"].str.match(r"^\d{5}").any(), "some non-products have a 5-digit StockCode"

remaining = remaining[~is_not_product].copy()

cleaning_log.append({
    "step": "non-product StockCodes removed",
    "rows_moved": len(non_products),
    "rows_remaining": len(remaining),
})

assert len(non_products) + len(remaining) == rows_before, "rows lost or counted twice"
assert remaining["StockCode"].str.match(r"^\d{5}").all(), "some non-product codes are still in remaining"
print(len(non_products), len(remaining))
non_products["StockCode"].value_counts().head(10)

2396 528884


StockCode
POST            1130
DOT              709
M                328
C2               142
DCGSSGIRL         13
BANK CHARGES      12
DCGSSBOY          11
GIFT_0001_20      10
GIFT_0001_10       9
GIFT_0001_30       8
Name: count, dtype: Int64

2,396 rows moved. The most common ones are POST (1,130 rows), DOT (709) and M (328), which are postage, fees and manual entries, not products. After the move, every `StockCode` left in `remaining` starts with five digits, and 528,884 rows are left.

### 4f. Zero price rows

Some rows have a `UnitPrice` of 0 but a positive `Quantity`. They look like free items or samples: they count as a transaction but bring in no money, which would distort Frequency and the price statistics. I move them out. This also covers the 12,540-unit invoice from notebook 01, so it doesn't need its own rule.

In [17]:
is_zero_price = (remaining["UnitPrice"] == 0) & (remaining["Quantity"] > 0)
print(is_zero_price.sum(), "rows with zero UnitPrice found")
remaining[is_zero_price][["InvoiceNo", "Description", "Quantity", "UnitPrice", "CustomerID"]].head(10)

1161 rows with zero UnitPrice found


,InvoiceNo,Description,Quantity,UnitPrice,CustomerID
622,536414,<NA>,56,0.0,<NA>
1970,536545,<NA>,1,0.0,<NA>
1971,536546,<NA>,1,0.0,<NA>
1972,536547,<NA>,1,0.0,<NA>
1987,536549,<NA>,1,0.0,<NA>
1988,536550,<NA>,1,0.0,<NA>
2024,536552,<NA>,1,0.0,<NA>
2025,536553,<NA>,3,0.0,<NA>
2026,536554,<NA>,23,0.0,<NA>
4348,536765,<NA>,19,0.0,<NA>


In [18]:
remaining[is_zero_price][["Description", "CustomerID"]].isnull().sum(), remaining[is_zero_price]["Quantity"].max()

(Description     584
 CustomerID     1128
 dtype: int64,
 np.int64(12540))

In [19]:
rows_before = len(remaining)

zero_price = remaining[is_zero_price].copy()

assert (zero_price["UnitPrice"] == 0).all(), "some zero price rows have a non-zero UnitPrice"
assert (zero_price["Quantity"] > 0).all(), "some zero price rows have a non-positive Quantity"

remaining = remaining[~is_zero_price].copy()

cleaning_log.append({
    "step": "zero price rows removed",
    "rows_moved": len(zero_price),
    "rows_remaining": len(remaining),
})

assert len(zero_price) + len(remaining) == rows_before, "rows lost or counted twice"
assert (remaining["UnitPrice"] > 0).all(), "some rows left in remaining have a non-positive UnitPrice"
assert (remaining["Quantity"] > 0).all(), "some rows left in remaining have a non-positive Quantity"
print(len(zero_price), len(remaining))

1161 527723


1,161 rows moved. Most of them have no `CustomerID` (1,128 of them) and 584 have no `Description`. The largest quantity is 12,540, which is the invoice I looked at in notebook 01, so the rule catches it. After this step, every row left in `remaining` has a positive `UnitPrice` and a positive `Quantity`, and 527,723 rows are left.

## 5. Check that every row is accounted for

Before going further I want to prove that no row got lost or counted twice. I turn the cleaning log into a table, build one `excluded` table with a reason for each row, and check that all the groups add up to the 541,909 rows of the raw data.

In [20]:
cleaning_summary = pd.DataFrame(cleaning_log)
cleaning_summary

,step,rows_moved,rows_remaining
0,bad debt invoices removed,3,541906
1,extreme order-and-cancel pairs removed,4,541902
2,cancellations separated,9286,532616
3,stock adjustments removed,1336,531280
4,non-product StockCodes removed,2396,528884
5,zero price rows removed,1161,527723


In [21]:
excluded = pd.concat([
    bad_debt.assign(reason="bad debt"),
    extreme_pairs.assign(reason="extreme pair"),
    stock_adjustments.assign(reason="stock adjustment"),
    non_products.assign(reason="non-product code"),
    zero_price.assign(reason="zero price")
])
print(excluded["reason"].value_counts())

reason
non-product code    2396
stock adjustment    1336
zero price          1161
extreme pair           4
bad debt               3
Name: count, dtype: int64


In [22]:
all_rows = pd.concat([excluded, cancellations, remaining])

assert all_rows.index.is_unique, "some rows appear in more than one table"
assert len(all_rows) == len(retail_data), "some rows are missing"
assert set(all_rows.index) == set(retail_data.index), "the row numbers do not match the raw data"
all_rows.shape

(541909, 9)

The six steps moved 14,186 rows out of the data in total, and 527,723 rows are left in `remaining`. Together with the cancellations, the excluded rows and `remaining`, the groups add up to 541,909, the same as the raw data, and no row appears in two tables. The `excluded` table has 4,900 rows, and the biggest reason is non-product code (2,396 rows).

## 6. Standardize descriptions

In notebook 01 I found that many stock codes have more than one `Description`. That means the name can't identify a product, but the `StockCode` can. So I give every code a single description: the one that appears most often for that code. Nothing is moved in this step, only the text of the column changes.

In [23]:
print(remaining["Description"].isnull().sum(), remaining["StockCode"].nunique())

0 3790


In [24]:
descriptions_per_code = remaining.groupby("StockCode")["Description"].nunique()
print(descriptions_per_code.max())
print((descriptions_per_code > 1).sum())

4
212


In [25]:
example_code = descriptions_per_code.idxmax()
print(example_code)

remaining[remaining["StockCode"] == example_code]["Description"].value_counts()

23196


Description
VINTAGE LEAF MAGNETIC NOTEPAD         281
RETRO LEAVES MAGNETIC NOTEPAD          22
RETO LEAVES MAGNETIC SHOPPING LIST      3
LEAVES MAGNETIC  SHOPPING LIST          2
Name: count, dtype: Int64

In [26]:
standard_description = remaining.groupby("StockCode")["Description"].agg(lambda x:x.value_counts().index[0])
print(len(standard_description))
print(standard_description.isnull().sum())

3790
0


In [27]:
new_description = remaining["StockCode"].map(standard_description)
changed_rows = (new_description != remaining["Description"]).sum()
print(changed_rows)
remaining["Description"] = new_description
assert remaining["Description"].notna().all(), "some description has NaN value"
assert remaining.groupby("StockCode")["Description"].nunique().max() == 1, "some StockCode has multiple Description"
assert len(remaining) == 527723, "rows were lost while standardizing descriptions"


6845


212 stock codes had more than one description, and one code had up to 4. Looking at the code with the most variants (23196), the names look like typos (`RETO` instead of `RETRO`, a double space) and older names of the same product, while the most common name appears in 281 rows. I use the most common description per code, which changed 6,845 rows. After the change, every `StockCode` has exactly one `Description`, there are no missing descriptions, and 527,723 rows are still in `remaining`.

## 7. Duplicates

Some rows are exact copies of another row: same invoice, same product, same quantity, same price and the same minute. I can't prove they are entry mistakes, but identical lines like this look much more like a double entry or a double scan than a real repeat purchase. I check how big they are first, and then I decide.

In [28]:
is_duplicate = remaining.duplicated()
print(is_duplicate.sum())
remaining[remaining.duplicated(keep=False)].sort_values(["InvoiceNo", "StockCode"]).head(10)

5223


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908,United Kingdom
565,536412,21448,12 DAISY PEGS IN WOOD BOX,2,2010-12-01 11:49:00,1.65,17920,United Kingdom
578,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920,United Kingdom


In [29]:
duplicate_revenue = (remaining.loc[is_duplicate]["Quantity"] * remaining.loc[is_duplicate]["UnitPrice"]).sum()
print(duplicate_revenue)
total_revenue = total_revenue = (remaining["Quantity"] * remaining["UnitPrice"]).sum()
print(total_revenue)
duplicate_pct = (duplicate_revenue / total_revenue) * 100
print(duplicate_pct)

24765.59
10025381.41
0.24702890580598869


In [30]:
original_duplicates = clean.loc[remaining.index].duplicated().sum()
print(original_duplicates)

5221


In [31]:
rows_before = len(remaining)

duplicates = remaining[is_duplicate].copy()
assert len(duplicates) == 5223, f"Assertion failed: Expected 5223 duplicates, got {len(duplicates)}"

remaining = remaining[~is_duplicate].copy()

cleaning_log.append({
    "step": "duplicate rows removed",
    "rows_moved": len(duplicates),
    "rows_remaining": len(remaining)
})

assert len(remaining) == rows_before - len(duplicates), "Assertion failed: Row count math does not match!"
assert not remaining.duplicated().any(), "Assertion failed: There are still duplicate rows in remaining!"

print(len(duplicates), len(remaining))

5223 522500


5,223 rows are exact duplicates, worth 24,765.59 in revenue, which is about 0.25% of the total. I drop them: if I'm wrong, the effect is tiny, and it would only touch the Monetary part of RFM, because Frequency counts unique invoices and is not affected. Before standardizing the descriptions there were only 5,221 duplicates, so the order of the steps mattered. After dropping them, 522,500 rows are left in `remaining`.

## 8. Missing CustomerID

About a quarter of the rows have no `CustomerID`. In the plan I decided to keep them, because they are still valid sales. Here I only measure how big the gap is and where it comes from, so I know what the RFM step will be built on. No rows are moved in this step.

In [32]:
is_missing_customer = remaining["CustomerID"].isnull()
print(is_missing_customer.sum())
missing_customer_pct = is_missing_customer.mean()* 100
print(missing_customer_pct)
unique_customer = remaining["CustomerID"].nunique()
print(unique_customer)

131354
25.139521531100478
4333


In [33]:
missing_by_country = (is_missing_customer.groupby(remaining["Country"]).mean()).sort_values(ascending=False)
(missing_by_country * 100).round(1).head(5)

Country
Hong Kong         100.0
Unspecified        45.5
United Kingdom     27.1
Israel             16.1
EIRE                8.3
Name: CustomerID, dtype: float64

In [34]:
revenue = remaining["Quantity"] * remaining["UnitPrice"]
missing_revenue_pct = (revenue[is_missing_customer].sum() / revenue.sum()) * 100
print(missing_revenue_pct)

15.095002719542524


In [35]:
invoices_without_customer = remaining.loc[is_missing_customer, "InvoiceNo"].nunique()
print(invoices_without_customer)
invoices_with_customer = remaining.loc[~is_missing_customer, "InvoiceNo"].nunique()
print(invoices_with_customer)
total_unique_invoices = remaining["InvoiceNo"].nunique()
print(total_unique_invoices)

assert invoices_with_customer + invoices_without_customer == total_unique_invoices, "some invoice is mixed"

1371
18400
19771


131,354 rows (25.14%) have no `CustomerID`, but they only make up about 15.1% of the revenue, so these rows are smaller on average. The gap is not random: Hong Kong has no customer at all, Unspecified is missing about 45.5% and the United Kingdom about 27.1%. The missing customers come in whole invoices (1,371 invoices) and never mix with identified customers inside the same invoice. I keep them in the sales table, and the RFM notebook will filter them out. That leaves 4,333 identified customers.

## 9. Final checks on the sales table

All the rules have been applied, so `remaining` is now my sales table. I give it its final name, `sales`, and check every rule with an `assert` so that nothing slipped through. I do the same for the cancellations table.

In [36]:
sales = remaining.copy()
sales.shape

(522500, 8)

In [37]:
assert (sales["Quantity"] > 0).all(), "some Quantity is not positive"
assert (sales["UnitPrice"] > 0).all(), "some UnitPrice is not positive"
assert sales["Description"].notna().all(), "some Description is missing"
assert sales["StockCode"].str.match(r"^\d{5}").all(), "some StockCode does not start with 5 digits"
assert not sales["InvoiceNo"].str.match(r"^[AC]").any(), "some InvoiceNo starts with 'A' or 'C'"
assert not sales.duplicated().any(), "duplicate rows still exist in sales"
assert sales.groupby("StockCode")["Description"].nunique().max() == 1, "some StockCode has more than one Description"
assert str(sales["CustomerID"].dtype) == "Int64", "CustomerID is not Int64"

In [38]:
assert (cancellations["Quantity"] < 0).all(), "some quantity has >= 0"
assert cancellations["InvoiceNo"].str.startswith("C").all(), "InvoiceNo that does not start with 'C'"

In [39]:
print(sales["InvoiceNo"].nunique())
print(sales["CustomerID"].nunique())
print(sales["StockCode"].nunique())
print(round((sales["Quantity"] * sales["UnitPrice"]).sum(), 2))
print(sales["InvoiceDate"].min())
print(sales["InvoiceDate"].max())

19771
4333
3790
10000615.82
2010-12-01 08:26:00
2011-12-09 12:50:00


All the checks pass. The `sales` table has 522,500 rows, 19,771 invoices, 4,333 customers and 3,790 stock codes, and it covers 10,000,615.82 in revenue. The dates run from 2010-12-01 to 2011-12-09. The `cancellations` table has 9,286 rows and all of them have a negative `Quantity`.

## 10. Final accounting

The duplicates were moved after my first accounting check, so I need to redo it. I rebuild the cleaning summary from the log, add the duplicates to the excluded table with their own reason, and prove again that the three final tables (sales, cancellations and excluded) cover all 541,909 raw rows exactly once.

In [40]:
cleaning_summary = pd.DataFrame(cleaning_log)
print(cleaning_summary)

assert list(cleaning_summary.columns) == ["step", "rows_moved", "rows_remaining"], (
    "Assertion failed: cleaning_log keys are not consistent!"
)

total_accounted = cleaning_summary["rows_moved"].sum() + len(sales)
assert total_accounted == len(retail_data), (
    f"Assertion failed: Row count mismatch! "
    f"Accounted ({total_accounted}) != Raw data ({len(retail_data)})"
)

                                     step  rows_moved  rows_remaining
0               bad debt invoices removed           3          541906
1  extreme order-and-cancel pairs removed           4          541902
2                 cancellations separated        9286          532616
3               stock adjustments removed        1336          531280
4          non-product StockCodes removed        2396          528884
5                 zero price rows removed        1161          527723
6                  duplicate rows removed        5223          522500


In [41]:
excluded_all = pd.concat([excluded, duplicates.assign(reason="duplicate")])
print(excluded_all["reason"].value_counts())

reason
duplicate           5223
non-product code    2396
stock adjustment    1336
zero price          1161
extreme pair           4
bad debt               3
Name: count, dtype: int64


In [42]:
all_rows_final = pd.concat([excluded_all, cancellations, sales])
assert all_rows_final.index.is_unique, "duplicate row indices found across split tables!"

assert len(all_rows_final) == len(retail_data), (
    f"row count mismatch! "
    f"Final ({len(all_rows_final)}) != Raw ({len(retail_data)})"
)

assert set(all_rows_final.index) == set(retail_data.index), "row indices do not match raw retail_data!"

In [43]:
row_accounting = pd.DataFrame({
    "rows": [len(sales), len(cancellations), len(excluded_all)]
}, index=["sales", "cancellations", "excluded"])

total_rows = row_accounting["rows"].sum()
row_accounting["share_%"] = ((row_accounting["rows"] / total_rows) * 100).round(2)

assert total_rows == len(retail_data), (
    f"Assertion failed: Total accounted rows ({total_rows}) != Raw data ({len(retail_data)})!"
)

print(row_accounting)

                 rows  share_%
sales          522500    96.42
cancellations    9286     1.71
excluded        10123     1.87


The seven steps moved 19,409 rows out of the main table, and together with the 522,500 rows in `sales` that adds up to the 541,909 raw rows. The `excluded_all` table has 10,123 rows, and the biggest reason is duplicate (5,223 rows), followed by non-product code (2,396 rows). Every raw row ends up in exactly one table: 96.42% in `sales`, 1.71% in `cancellations` and 1.87% in `excluded_all`.

## 11. Save the results

I save the three tables to `data/processed/`: `sales_clean.csv` (the real sales used for analysis), `cancellations.csv` (kept for later) and `excluded_rows.csv` (every excluded row with its reason, for auditing). The raw row number goes into a column called `raw_index`, so any row can be traced back to the raw file. The folder is ignored by git, so anyone who clones the repo can recreate these files by running this notebook.

In [44]:
PROCESSED_DIR = "../data/processed"

sales.to_csv(f"{PROCESSED_DIR}/sales_clean.csv", index=True, index_label="raw_index")
cancellations.to_csv(f"{PROCESSED_DIR}/cancellations.csv", index=True, index_label="raw_index")
excluded_all.to_csv(f"{PROCESSED_DIR}/excluded_rows.csv", index=True, index_label="raw_index")

In [45]:
check_sales = pd.read_csv(
    f"{PROCESSED_DIR}/sales_clean.csv", 
    index_col="raw_index", 
    parse_dates=["InvoiceDate"]
)

check_cancellations = pd.read_csv(
    f"{PROCESSED_DIR}/cancellations.csv", 
    index_col="raw_index", 
    parse_dates=["InvoiceDate"]
)

check_excluded = pd.read_csv(
    f"{PROCESSED_DIR}/excluded_rows.csv", 
    index_col="raw_index", 
    parse_dates=["InvoiceDate"]
)

assert len(check_sales) == len(sales), "Sales row count mismatch!"
assert len(check_cancellations) == len(cancellations), "Cancellations row count mismatch!"
assert len(check_excluded) == len(excluded_all), "Excluded rows count mismatch!"

assert check_sales.index.equals(sales.index), "Sales index mismatch!"
assert check_cancellations.index.equals(cancellations.index), "Cancellations index mismatch!"
assert check_excluded.index.equals(excluded_all.index), "Excluded index mismatch!"

assert check_sales.shape == (522500, 8), f"Unexpected sales shape: {check_sales.shape}"
assert check_cancellations.shape == (9286, 8), f"Unexpected cancellations shape: {check_cancellations.shape}"
assert check_excluded.shape == (10123, 9), f"Unexpected excluded shape: {check_excluded.shape}"

print("check_sales.shape:", check_sales.shape)
print("check_cancellations.shape:", check_cancellations.shape)
print("check_excluded.shape:", check_excluded.shape)
print("All assertions passed!")

check_sales.shape: (522500, 8)
check_cancellations.shape: (9286, 8)
check_excluded.shape: (10123, 9)
All assertions passed!


In [46]:
check_sales.dtypes

InvoiceNo               int64
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object

All three files were saved and reloaded: `sales_clean.csv` has 522,500 rows, `cancellations.csv` has 9,286 rows and `excluded_rows.csv` has 10,123 rows, and the row numbers survive the round trip. One thing to remember: a CSV does not store column types, so after reading it again `CustomerID` comes back as a float (because of the missing values) and `InvoiceNo` in `sales` comes back as an integer. The next notebook has to convert `CustomerID` to `Int64` again.

## 12. Summary

Every one of the 541,909 raw rows ended up in exactly one table:

| Table | File | Rows | Share | What it is |
|---|---|---|---|---|
| `sales` | `sales_clean.csv` | 522,500 | 96.42% | Real sales, used for the analysis |
| `cancellations` | `cancellations.csv` | 9,286 | 1.71% | Cancelled invoices, kept to adjust what a customer really spent |
| `excluded_all` | `excluded_rows.csv` | 10,123 | 1.87% | Everything else, with a reason for each row |

The excluded rows are 5,223 duplicates, 2,396 non-product codes, 1,336 stock adjustments, 1,161 zero-price rows, 4 extreme order-and-cancel rows and 3 bad debt rows.

Things to remember for the RFM step:
- 131,354 rows (25.14%) in `sales` have no `CustomerID`, but they only make up about 15.1% of the revenue. They stay in `sales`, and the RFM notebook filters them out, which leaves 4,333 customers.
- The data stops on 9 December 2011, so the reference date for Recency has to be chosen with that in mind.
- The cancellations are in their own table, so I still have to decide whether they reduce the Monetary value.
- `CustomerID` and `InvoiceNo` change type when the CSV is read, so I convert them again in the next notebook.
- The raw row number is stored in `raw_index`, so any row can be traced back to the raw file.

Next is `03_feature_engineering.ipynb`, where I compute Recency, Frequency and Monetary for each customer.